# Notebook 2 — Strands Agents with Skills

This classroom-ready notebook builds on Notebook 1.

Notebook 1 focused on:

- Agent loop
- Prompts
- Tools
- Memory
- Structured output
- Conversation and session basics

Notebook 2 focuses on **Skills**.

## What this notebook covers

By the end of this notebook, learners should understand:

1. What a Skill is
2. Why Skills are different from normal tools
3. How Skills reduce system prompt bloat
4. How to create Skills programmatically
5. How to attach Skills to an agent using `AgentSkills`
6. How to inspect available Skills
7. How an agent activates Skills on demand
8. How to create directory-based `SKILL.md` Skills
9. How Skills can work with tools
10. How Skills can work with memory
11. How learners can design their own Skills for PRD review, test generation, and onboarding support


## Install packages

Run the below cell only if the packages are not already available in the notebook kernel.

In [ ]:
# Uncomment if needed
!pip install -q boto3 pydantic==2.9 
!pip install -q -U strands-agents strands-agents-tools

In [ ]:
import strands
print(strands.__file__)

## Imports

The below code imports the packages required for Strands agents, Bedrock model access, tools, and Skills.

The Skills import is written with a fallback because some SDK versions expose Skills from different locations.

In [ ]:
import boto3
from pathlib import Path
from datetime import datetime
from typing import Optional

from pydantic import BaseModel, Field

from strands import Agent, tool
from strands.models import BedrockModel
from strands.vended_plugins.skills.agent_skills import AgentSkills
from strands.vended_plugins.skills.skill import Skill


## Bedrock configuration

The below code configures the Bedrock model used by the agents in this notebook.

Change `MODEL_ID` only if your classroom environment uses a different Bedrock model.

In [ ]:
AWS_REGION = boto3.session.Session().region_name or "us-east-1"

# Common classroom-friendly models:
# - amazon.nova-lite-v1:0
# - amazon.nova-micro-v1:0
# - amazon.nova-pro-v1:0

MODEL_ID = "amazon.nova-lite-v1:0"

bedrock_model = BedrockModel(
    model_id=MODEL_ID,
    region_name=AWS_REGION,
    temperature=0.2,
)

print("Region:", AWS_REGION)
print("Model:", MODEL_ID)

## Helper function

The below code creates a small helper function to print agent responses cleanly.

In [ ]:
def show_response(title, response):
    print("\n")
    print("=" * 80)
    print(title)
    print("=" * 80)
    print(response)

# Part A — What is a Skill?

A **Skill** is a reusable instruction package.

It is useful when an agent may need many specialized capabilities, but we do not want to put all instructions inside one long system prompt.

## Simple difference

| Concept | Purpose |
|---|---|
| System prompt | Always-on behavior instructions |
| Tool | Action the agent can execute |
| Skill | Specialized instructions loaded only when relevant |

## Why Skills matter

Without Skills, a single agent may have a huge system prompt containing instructions for email writing, policy simplification, PRD review, code review, test generation, and more.

With Skills, the agent first sees only lightweight metadata such as the Skill name and description. The full instructions are loaded only when the agent activates that Skill.

# Example 1 — Agent with one programmatic Skill

In this example, the agent has one Skill: **policy-simplifier**.

The Skill tells the agent how to rewrite complex HR or admin policy text into employee-friendly language.


## Create an AgentSkills plugin

The below code creates a Skill programmatically using `Skill(...)`.

In [ ]:
policy_simplifier_skill = Skill(
    name="policy-simplifier",
    description="Rewrite HR or admin policy text into simple employee-friendly language.",
    instructions="""
You are the Policy Simplifier Skill.

When this skill is used:
1. Rewrite complex policy text in simple language.
2. Keep the original meaning.
3. Avoid legal jargon unless required.
4. Use this format:
   - Simple version
   - Key rules
   - Employee action needed
""",
)

policy_skill_plugin = AgentSkills(skills=[policy_simplifier_skill])

## Create an agent with the Policy Skill

The below code creates an agent that can use the Policy Simplifier Skill.

In [ ]:
policy_agent = Agent(
    model=bedrock_model,
    plugins=[policy_skill_plugin],
    system_prompt="""
You are a helpful classroom demo agent.

Use the available skill when the user asks you to simplify HR, admin, or policy text.
Keep answers concise.
""",
)

## Test the Policy Skill agent

The below code asks the agent to simplify a policy statement.

In [ ]:
policy_text = """
Employees may work remotely up to two days per week subject to manager approval,
business continuity requirements, data security obligations, and client confidentiality constraints.
Remote work privileges may be revoked if performance, availability, or compliance expectations are not met.
"""

response = policy_agent(
    f"Please simplify this policy for employees:\n\n{policy_text}"
)

show_response("Example 1 — Policy Skill Output", response)

## Inspect activated Skills

After the agent runs, we can inspect which Skills were activated.

This is useful for teaching because learners can see that the Skill was not just text in the prompt; it was activated by the agent.

In [ ]:
activated = policy_skill_plugin.get_activated_skills(policy_agent)
print("Activated skills:", activated)

# Example 2 — Agent with multiple Skills

In this example, the same agent has two Skills:

1. **email-writer** — writes professional emails
2. **summary-writer** — creates concise summaries

The agent decides which Skill is relevant for each task.

## Create two Skills

The below code creates two different Skills for the same agent.

In [ ]:
email_writer_skill = Skill(
    name="email-writer",
    description="Write clear, polite, professional business emails.",
    instructions="""
You are the Email Writer Skill.

When this skill is used:
1. Write a clear subject line.
2. Keep the email polite and professional.
3. Use short paragraphs.
4. End with a clear next step.
""",
)

summary_writer_skill = Skill(
    name="summary-writer",
    description="Summarize long text into concise business points.",
    instructions="""
You are the Summary Writer Skill.

When this skill is used:
1. Identify the main message.
2. Extract only important points.
3. Avoid unnecessary detail.
4. Use a short bullet list.
""",
)

communication_skill_plugin = AgentSkills(
    skills=[email_writer_skill, summary_writer_skill]
)

print("Available communication skills:")
for skill in communication_skill_plugin.get_available_skills():
    print("-", skill.name, ":", skill.description)

## Create a communication agent with multiple Skills

The below code creates an agent that can use both communication Skills.

In [ ]:
communication_agent = Agent(
    model=bedrock_model,
    plugins=[communication_skill_plugin],
    system_prompt="""
You are a communication assistant.

Use the most relevant available skill based on the user's request.
Do not over-explain which skill was used unless asked.
""",
)

## Test the Email Writer Skill

The below code asks the agent to draft a short email.

In [ ]:
response = communication_agent(
    "Write a short email to the ADP team saying the pre-read material is attached and learners should complete it before the session."
)

show_response("Example 2A — Email Skill Output", response)

try:
    print("Activated skills:", communication_skill_plugin.get_activated_skills(communication_agent))
except Exception:
    pass

## Test the Summary Writer Skill

The below code asks the same agent to summarize a paragraph.

In [ ]:
long_text = """
The training program is designed to help software engineers understand how generative AI can support the software development lifecycle.
The program includes prompt engineering, code generation, test case generation, defect analysis, documentation support, and agent-based workflows.
Learners will complete hands-on labs using cloud-hosted notebooks and will compare manual outputs with AI-assisted outputs.
"""

response = communication_agent(
    f"Summarize this text into concise business points:\n\n{long_text}"
)

show_response("Example 2B — Summary Skill Output", response)

try:
    print("Activated skills:", communication_skill_plugin.get_activated_skills(communication_agent))
except Exception:
    pass

# Example 3 — Directory-based `SKILL.md` Skill

So far, Skills were created directly in Python.

In real projects, Skills are often stored as folders with a `SKILL.md` file.

This makes them easier to version, review, and share.

## Create a local Skill folder

The below code creates a classroom Skill folder inside the notebook environment.

Folder structure:

```text
skills/
└── test-case-writer/
    └── SKILL.md
```

In [ ]:
skills_root = Path("./skills")
test_case_skill_dir = skills_root / "test-case-writer"
test_case_skill_dir.mkdir(parents=True, exist_ok=True)

skill_md = """---
name: test-case-writer
description: Generate concise functional test cases from a requirement.
allowed-tools: count_test_cases
---

# Test Case Writer Skill

When asked to generate test cases:

1. Read the requirement carefully.
2. Generate clear test cases.
3. Use IDs such as TC-001, TC-002, TC-003.
4. Include positive, negative, and edge cases.
5. Keep each test case to one line unless more detail is required.
"""

(test_case_skill_dir / "SKILL.md").write_text(skill_md, encoding="utf-8")

print("Created:", test_case_skill_dir / "SKILL.md")
print((test_case_skill_dir / "SKILL.md").read_text(encoding="utf-8"))

## Load the directory-based Skill

The below code loads the Skill from the directory.

In [ ]:
directory_skill_plugin = AgentSkills(skills=str(test_case_skill_dir))

print("Available directory skills:")
for skill in directory_skill_plugin.get_available_skills():
    print("-", skill.name, ":", skill.description)

## Add a supporting tool

Skills provide instructions.

Tools provide actions.

The `allowed-tools` field in `SKILL.md` documents which tools the Skill expects, but the agent must still be given those tools separately.

In [ ]:
@tool
def count_test_cases(test_cases_text: str) -> str:
    """Count test cases by counting lines that start with TC-."""
    count = sum(
        1 for line in test_cases_text.splitlines()
        if line.strip().upper().startswith("TC-")
    )
    return f"Detected {count} test cases."

## Create an agent with directory Skill + tool

The below code creates an agent that has both the Skill and the supporting tool.

In [ ]:
test_case_agent = Agent(
    model=bedrock_model,
    plugins=[directory_skill_plugin],
    tools=[count_test_cases],
    system_prompt="""
You are a QA assistant.

Use the available test case writing skill when the user asks for test cases.
After generating test cases, use the tool to count them.
""",
)

requirement = """
Users should be able to reset their password using a one-time password sent to their registered email address.
The OTP should expire after 10 minutes.
"""

response = test_case_agent(
    f"Generate test cases for this requirement and count them:\n\n{requirement}"
)

show_response("Example 3 — Directory Skill + Tool Output", response)

try:
    print("Activated skills:", directory_skill_plugin.get_activated_skills(test_case_agent))
except Exception:
    pass

# Example 4 — Skill + Tool + Memory

This example combines:

| Component | Used here? | Purpose |
|---|---:|---|
| Brain | Yes | LLM reasoning |
| Skill | Yes | Customer support response style |
| Tool | Yes | Save, recall, and calculate |
| Memory | Yes | Store customer preferences |

## Create memory and tools

The below code creates:

- one memory dictionary
- one tool to save memory
- one tool to read memory
- one tool to calculate discounted price

In [ ]:
customer_memory = {}


@tool
def save_customer_preference(customer_name: str, preference: str) -> str:
    """Save a customer preference in memory."""
    customer_memory[customer_name.lower()] = preference
    return f"Saved preference for {customer_name}: {preference}"


@tool
def read_customer_preference(customer_name: str) -> str:
    """Read a customer preference from memory."""
    preference = customer_memory.get(customer_name.lower())
    if preference is None:
        return f"No preference found for {customer_name}."
    return f"{customer_name}'s preference is: {preference}"


@tool
def calculate_discounted_price(price: float, discount_percent: float) -> str:
    """Calculate final price after applying a discount percentage."""
    final_price = price * (1 - discount_percent / 100)
    return f"Price: {price:.2f}, Discount: {discount_percent:.1f}%, Final price: {final_price:.2f}"

## Create a Customer Support Skill

The below code creates a Skill that tells the agent how to respond to customer requests.

In [ ]:
customer_support_skill = Skill(
    name="customer-support-advisor",
    description="Handle customer support requests using a polite, concise, preference-aware response style.",
    instructions="""
You are the Customer Support Advisor Skill.

When this skill is used:
1. Be polite and concise.
2. If customer preference is needed, use memory tools.
3. If price calculation is needed, use calculation tools.
4. Do not invent customer preferences.
5. End with a clear next step or confirmation.
""",
)

customer_skill_plugin = AgentSkills(skills=[customer_support_skill])

## Create an agent with Skill + Tool + Memory

The below code creates the customer support agent.

In [ ]:
customer_agent = Agent(
    model=bedrock_model,
    plugins=[customer_skill_plugin],
    tools=[
        save_customer_preference,
        read_customer_preference,
        calculate_discounted_price,
    ],
    system_prompt="""
You are a customer support agent for classroom demonstration.

Use the available customer support skill when the task involves customer preferences,
customer replies, or discount-related support.
""",
)

## Test memory saving

The below code asks the agent to remember a customer's preference.

In [ ]:
response = customer_agent(
    "Customer name is Ravi. Remember that Ravi prefers short answers and usually asks for a 15% discount."
)

show_response("Example 4A — Save Memory Output", response)
print("\nCurrent memory dictionary:", customer_memory)

## Test memory + calculation

The below code asks the agent to use Ravi's remembered discount preference and calculate the final price.

In [ ]:
response = customer_agent(
    "For Ravi, calculate the final price for a product priced at 3200 using his usual discount preference. Then write a short customer reply."
)

show_response("Example 4B — Skill + Tool + Memory Output", response)

# Example 5 — Skill + structured output

Skills can guide how the agent thinks.

Structured output can make the final answer machine-readable.

This is useful when the output must be consumed by dashboards, evaluation scripts, or downstream workflows.

# Skill design checklist

Use this checklist when designing Skills.

| Design point | Good practice |
|---|---|
| Name | Use short lowercase names, such as `prd-reviewer` |
| Description | Make it clear when the agent should use the Skill |
| Instructions | Give step-by-step behavior |
| Output format | Specify the expected structure |
| Tools | Mention expected tools, but also provide those tools to the agent |
| Resources | Put reusable references under `references/`, scripts under `scripts/`, and templates under `assets/` |
| Scope | Use Skills for specialized instructions, not for every small task |

# Classroom Activity 1 — Create a PRD Review Skill

## Task

Create a new Skill called `prd-reviewer`.

## Expected behavior

The agent should review a Product Requirements Document and identify:

1. Missing user personas
2. Missing acceptance criteria
3. Missing non-functional requirements
4. Risks and dependencies
5. Suggested improvements

## learner instruction

Complete the below code and test the agent.

In [ ]:
# learner Activity 1: Create a PRD Review Skill
# Fill this
prd_reviewer_skill = Skill(
   ------
)

###
learner_prd_plugin = AgentSkills(skills=[prd_reviewer_skill])

learner_prd_agent = Agent(
    model=bedrock_model,
    plugins=[learner_prd_plugin],
    system_prompt="You are a PRD review assistant.",
)

sample_prd = """
Feature: Password reset using email OTP

Users should be able to reset their password if they forget it.
The system should send an OTP to the user's registered email address.
After OTP verification, the user can create a new password.
"""

response = learner_prd_agent(f"Review this PRD:\n\n{sample_prd}")
show_response("learner Activity 1 Output", response)

# Classroom Activity 2 — Create a Skill + Tool Agent

## Task

Create an agent that uses:

- one Skill for **test case generation**
- one tool to count the number of generated test cases

## learner instruction

Complete the Skill instructions and run the test prompt.

In [ ]:
# learner Activity 2: Skill + Tool Agent


# Wrap-up

In this notebook, learners saw that Skills are not the same as tools.

| Capability | What it gives the agent |
|---|---|
| Tool | Ability to perform an action |
| Skill | Specialized instructions loaded on demand |
| Memory | Ability to store and recall facts |
| Structured output | Machine-readable response format |

## Recommended sequence

1. Start with system prompts for simple behavior.
2. Add tools when the agent must act or calculate.
3. Add memory when the agent must remember facts.
4. Add Skills when the agent has many specialized instruction sets.